In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP


In [2]:
future_df = pd.read_csv("/Users/rhombus19/projects/eleo/future_features_dataset.csv")
train_df = pd.read_csv("/Users/rhombus19/projects/eleo/forecast_dataset.csv")

In [3]:
train_df["date"] = pd.to_datetime(train_df["date"])
train_df = train_df.sort_values(["SKU", "date"]).reset_index(drop=True)


In [5]:
train_df

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00,11-20,3,11,False
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00,11-21,4,11,False
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00,11-22,5,11,False
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00,11-23,6,11,False


In [6]:
sku = "9101dx"   # example

sku_hist = train_df[train_df["SKU"] == sku].copy()
sku_hist = sku_hist.sort_values("date").reset_index(drop=True)

# sanity check: if no sales at all, skip this SKU
if sku_hist["qty"].sum() == 0:
    raise ValueError("No non-zero sales for this SKU – cannot fit a useful ZINB.")


In [7]:
sku_hist

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,2025-11-20,9101dx,0.0,1295.228583,2.3,0.0,59.0,True,0.20,11-20,3,11,False
665,2025-11-21,9101dx,0.0,1295.228583,-1.0,0.0,136.0,True,0.20,11-21,4,11,False
666,2025-11-22,9101dx,0.0,1295.228583,-4.2,0.0,243.0,True,0.20,11-22,5,11,False
667,2025-11-23,9101dx,0.0,1295.228583,-2.6,2.5,155.0,True,0.20,11-23,6,11,False


In [10]:
# convert to numeric types we like
for col in ["is_holiday"]:
    sku_hist[col] = sku_hist[col].astype(int)

num_cols = ["tavg", "prcp", "tsun", "price_per_unit", "sale_percent"]
cat_cols = ["day_of_week", "month"]  # already ints in your data
bin_cols = ["is_holiday"]

# ----- design matrix for count process -----
X = sku_hist[num_cols + cat_cols + bin_cols].copy()

# one-hot encode categorical calendar fields
X = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True  # avoid dummy trap
)

# add intercept
X = sm.add_constant(X, has_constant="add")

# target
y = sku_hist["qty"].astype(int).to_numpy()

# ----- design matrix for zero-inflation (logit) part -----
Z = sku_hist[["sale_percent", "is_holiday"]].copy()
Z = sm.add_constant(Z, has_constant="add")


In [17]:
print(X.dtypes)
print(Z.dtypes)
print(X.head())


const             float64
tavg              float64
prcp              float64
tsun              float64
price_per_unit    float64
sale_percent      float64
is_holiday          int64
day_of_week_1        bool
day_of_week_2        bool
day_of_week_3        bool
day_of_week_4        bool
day_of_week_5        bool
day_of_week_6        bool
month_2              bool
month_3              bool
month_4              bool
month_5              bool
month_6              bool
month_7              bool
month_8              bool
month_9              bool
month_10             bool
month_11             bool
month_12             bool
dtype: object
const           float64
sale_percent    float64
is_holiday        int64
dtype: object
   const  tavg  prcp   tsun  price_per_unit  sale_percent  is_holiday  \
0    1.0   9.0   1.2    0.0     1522.660000          0.24           0   
1    1.0   3.0   0.0  510.0     1295.228583          0.24           0   
2    1.0   3.7   0.0  516.0     1295.228583          0.24

In [16]:
zinb_model = ZeroInflatedNegativeBinomialP(
    endog=y,
    exog=np.asarray(X),
    exog_infl=Z,
    inflation="logit"  # logit model for the extra zeros
)

zinb_res = zinb_model.fit(maxiter=200, method="lbfgs", disp=False)
print(zinb_res.summary())


TypeError: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''